### 表格数据

In [2]:
import os
import json
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

DATA_ROOT = r"c:\Users\16960\Desktop\期末论文\三模态数据库建立说明\scripts\Aeolus_V2\dataset\Flight_Tabular"
OUT_DIR = r"c:\Users\16960\Desktop\期末论文\模型搭建\data\数据集划分\表格"
YEAR = 2024
DELAY_THRESHOLD = 15

# 读取原始表格数据
files = sorted(glob.glob(os.path.join(DATA_ROOT, str(YEAR), "**", "*.csv"), recursive=True))
if not files:
    raise FileNotFoundError(f"未找到数据文件: {DATA_ROOT}\\{YEAR}")

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# 构造标签
df["label"] = (df["DEP_DELAY"] > DELAY_THRESHOLD).astype(int)

# 特征分组
cat_cols = [
    "OP_CARRIER", "OP_CARRIER_FL_NUM",
    "FL_YEAR", "FL_MONTH", "FL_DAY", "FL_WEEK",
    "ORIGIN_INDEX", "DEST_INDEX",
]
cont_cols = [
    "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN", "CRS_ELAPSED_TIME", "FLIGHTS",
    "O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD",
    "O_LATITUDE", "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE",
]

CN = {
    "OP_CARRIER": "承运人",
    "OP_CARRIER_FL_NUM": "航班号",
    "FL_YEAR": "年份",
    "FL_MONTH": "月份",
    "FL_DAY": "日期",
    "FL_WEEK": "星期",
    "ORIGIN_INDEX": "出发机场",
    "DEST_INDEX": "到达机场",
    "CRS_DEP_TIME_MIN": "计划出发时间(分钟)",
    "CRS_ARR_TIME_MIN": "计划到达时间(分钟)",
    "CRS_ELAPSED_TIME": "计划飞行时长",
    "FLIGHTS": "航班频次",
    "O_TEMP": "出发地气温",
    "O_PRCP": "出发地降水",
    "O_WSPD": "出发地风速",
    "D_TEMP": "到达地气温",
    "D_PRCP": "到达地降水",
    "D_WSPD": "到达地风速",
    "O_LATITUDE": "出发地纬度",
    "O_LONGITUDE": "出发地经度",
    "D_LATITUDE": "到达地纬度",
    "D_LONGITUDE": "到达地经度",
    "label": "是否延误",
}

X = df[cat_cols + cont_cols].copy()
y = df["label"].to_numpy()

# 先按时间切分，再只用训练集拟合预处理参数，避免数据泄漏
date_str = (
    df["FL_YEAR"].astype(str).str.zfill(4) + "-"
    + df["FL_MONTH"].astype(str).str.zfill(2) + "-"
    + df["FL_DAY"].astype(str).str.zfill(2)
)
unique_dates = sorted(date_str.unique())
n = len(unique_dates)
cut1, cut2 = int(n * 0.6), int(n * 0.8)

train_dates = set(unique_dates[:cut1])
val_dates = set(unique_dates[cut1:cut2])
test_dates = set(unique_dates[cut2:])

masks = {
    "train": date_str.isin(train_dates).to_numpy(),
    "val": date_str.isin(val_dates).to_numpy(),
    "test": date_str.isin(test_dates).to_numpy(),
}

split_frames = {name: X.loc[masks[name]].copy() for name in masks}
split_labels = {name: y[masks[name]] for name in masks}

# 缺失值填补：连续特征补 0，类别特征使用训练集众数
cat_fill_values = {}
for c in cat_cols:
    train_mode = split_frames["train"][c].mode(dropna=True)
    cat_fill_values[c] = train_mode.iloc[0] if not train_mode.empty else "UNKNOWN"

for frame in split_frames.values():
    frame[cont_cols] = frame[cont_cols].fillna(0)
    for c in cat_cols:
        frame[c] = frame[c].fillna(cat_fill_values[c])

# 仅基于训练集建立类别映射，验证/测试中未见类别统一映射到 UNK
cat_mappings = {}
for c in cat_cols:
    classes = sorted(split_frames["train"][c].astype(str).unique().tolist())
    class_to_idx = {v: i for i, v in enumerate(classes)}
    unk_idx = len(classes)

    cat_mappings[c] = {
        "classes": classes,
        "unknown_index": unk_idx,
    }

    for frame in split_frames.values():
        values = frame[c].astype(str)
        frame[c] = values.map(class_to_idx).fillna(unk_idx).astype(np.int32)

# 连续特征标准化：仅使用训练集拟合
scaler = StandardScaler()
split_frames["train"][cont_cols] = scaler.fit_transform(split_frames["train"][cont_cols])
for name in ["val", "test"]:
    split_frames[name][cont_cols] = scaler.transform(split_frames[name][cont_cols])

# 保存
os.makedirs(OUT_DIR, exist_ok=True)
for name in ["train", "val", "test"]:
    np.savez(
        os.path.join(OUT_DIR, f"tabular_{name}.npz"),
        X=split_frames[name][cat_cols + cont_cols].values.astype(np.float32),
        y=split_labels[name],
    )

info = {
    "feature_cols": cat_cols + cont_cols,
    "cat_cols": cat_cols,
    "cont_cols": cont_cols,
    "feature_name_map": CN,
    "cat_fill_values": {k: str(v) for k, v in cat_fill_values.items()},
    "cat_mappings": cat_mappings,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "split_stats": {
        name: {
            "rows": int(len(split_frames[name])),
            "positive_rate": float(split_labels[name].mean()),
        }
        for name in ["train", "val", "test"]
    },
}

with open(os.path.join(OUT_DIR, "preprocess_info.json"), "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

# 导出一份汉化字段示例，便于人工查看
xlsx_dir = os.path.join(OUT_DIR, "表格")
os.makedirs(xlsx_dir, exist_ok=True)
df[cat_cols + cont_cols + ["label"]].sample(n=100000, random_state=42).rename(
    columns=CN
).to_excel(os.path.join(xlsx_dir, "tabular_sample_100k.xlsx"), index=False)

print(json.dumps(info["split_stats"], ensure_ascii=False, indent=2))

{
  "train": {
    "rows": 4064344,
    "positive_rate": 0.21681530894038498
  },
  "val": {
    "rows": 1402119,
    "positive_rate": 0.15935024060012024
  },
  "test": {
    "rows": 1387575,
    "positive_rate": 0.16066158586022378
  }
}


### 链数据

In [2]:
import os, glob
import numpy as np
import torch

CHAIN_ROOT = r"c:\Users\16960\Desktop\期末论文\三模态数据库建立说明\scripts\Aeolus_V2\dataset\Flight_Chain"
OUT_DIR = r"C:\Users\16960\Desktop\期末论文\模型搭建\data\数据集划分\链数据"
YEAR = 2024

files = sorted(glob.glob(os.path.join(CHAIN_ROOT, str(YEAR), "**", "*.pt"), recursive=True))

all_dense, all_sparse, all_labels, all_vlen = [], [], [], []
dates = []

for f in files:
    d = torch.load(f, weights_only=True)
    n = d['dense_feat'].shape[0]
    all_dense.append(d['dense_feat'].numpy())
    all_sparse.append(d['sparse_feat'].numpy())
    all_labels.append(d['labels'].numpy().reshape(n, -1))
    all_vlen.append(d['valid_len'].numpy())
    ymd = os.path.basename(f).replace('flight_chain_', '').replace('.pt', '')
    dates.extend([ymd] * n)

dense = np.concatenate(all_dense)
sparse = np.concatenate(all_sparse)
labels = np.concatenate(all_labels)
vlen = np.concatenate(all_vlen)

unique_dates = sorted(set(dates))
n_d = len(unique_dates)
c1, c2 = int(n_d * 0.6), int(n_d * 0.8)
train_d = set(unique_dates[:c1])
val_d = set(unique_dates[c1:c2])
test_d = set(unique_dates[c2:])

date_arr = np.array(dates)

os.makedirs(OUT_DIR, exist_ok=True)

for name, dset in [('train', train_d), ('val', val_d), ('test', test_d)]:
    mask = np.isin(date_arr, list(dset))
    nd, nd2, nl, nv = dense[mask], sparse[mask], labels[mask], vlen[mask]
    N = nd.shape[0]
    np.savez(os.path.join(OUT_DIR, f"chain_{name}.npz"),
             dense=nd.reshape(N, -1), sparse=nd2.reshape(N, -1),
             labels=nl, vlen=nv)

info = {
    'dense_dim': 7,
    'sparse_dim': 9,
    'max_chain': 6,
    'sparse_names': ['FL_MONTH','FL_WEEK','CAH','CDH','OI','DI','OC_ENC','FN_ENC','TE'],
}
import json
with open(os.path.join(OUT_DIR, "chain_info.json"), 'w') as f:
    json.dump(info, f, indent=2)

### 网格数据

In [ ]:
### Network data
import os, glob, json
from pathlib import Path
import dgl
import torch

ROOT = Path.home() / "Desktop" / "\u671f\u672b\u8bba\u6587"
NETWORK_ROOT = ROOT / "\u4e09\u6a21\u6001\u6570\u636e\u5e93\u5efa\u7acb\u8bf4\u660e" / "scripts" / "Aeolus_V2" / "dataset" / "Flight_Network"
OUT_DIR = ROOT / "\u6a21\u578b\u642d\u5efa" / "data" / "\u6570\u636e\u96c6\u5212\u5206" / "\u7f51\u7edc\u6570\u636e"
YEAR = 2024
LABEL_THRESHOLD_MINUTES = 15

files = sorted(glob.glob(str(NETWORK_ROOT / str(YEAR) / "**" / "*.dgl"), recursive=True))
if not files:
    raise FileNotFoundError(f"No network graphs found under {NETWORK_ROOT / str(YEAR)}")

dates = [os.path.basename(f).replace('flight_network_', '').replace('.dgl', '') for f in files]
unique_dates = sorted(set(dates))
n_d = len(unique_dates)
c1, c2 = int(n_d * 0.6), int(n_d * 0.8)
splits = {
    'train': set(unique_dates[:c1]),
    'val': set(unique_dates[c1:c2]),
    'test': set(unique_dates[c2:]),
}

os.makedirs(OUT_DIR, exist_ok=True)
info = {
    'feat_dim': None,
    'year': YEAR,
    'label_unit': 'minutes',
    'label_threshold_minutes': LABEL_THRESHOLD_MINUTES,
    'splits': {},
}

for name, dset in splits.items():
    g_list = []
    node_count = 0
    edge_count = 0
    pos_count = 0
    total_count = 0

    for f, d in zip(files, dates):
        if d not in dset:
            continue
        graphs, _ = dgl.load_graphs(f)
        g = graphs[0]
        g.ndata['feat'] = torch.nan_to_num(g.ndata['feat'].float(), nan=0.0, posinf=0.0, neginf=0.0)
        g.ndata['label'] = torch.nan_to_num(g.ndata['label'].float(), nan=0.0, posinf=0.0, neginf=0.0)
        g.ndata['label_bin'] = (g.ndata['label'] > LABEL_THRESHOLD_MINUTES).float()
        g_list.append(g)

        node_count += g.num_nodes()
        edge_count += g.num_edges()
        pos_count += int(g.ndata['label_bin'].sum().item())
        total_count += g.ndata['label_bin'].numel()
        if info['feat_dim'] is None:
            info['feat_dim'] = int(g.ndata['feat'].shape[1])

    out_path = OUT_DIR / f"network_{name}.dgl"
    dgl.save_graphs(str(out_path), g_list)
    info['splits'][name] = {
        'graphs': len(g_list),
        'nodes': node_count,
        'edges': edge_count,
        'positive': pos_count,
        'total': total_count,
        'positive_rate': (pos_count / total_count) if total_count else 0.0,
    }
    print(f"{name}: {len(g_list)} graphs, {node_count:,} nodes, pos_rate={info['splits'][name]['positive_rate']:.4f}")

with open(OUT_DIR / "network_info.json", 'w', encoding='utf-8') as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print(json.dumps(info, ensure_ascii=False, indent=2))
